# CTNSG Master Kaggle Training Notebook
This notebook orchestrates the end-to-end training pipeline for the Canonical Tractable Neuro-Symbolic Generation Framework.
It assumes execution on Kaggle with dual T4 or P100 GPUs (13-16GB VRAM).

**Note:** This notebook is configured for full training.

In [ ]:
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/Borisz42/CTNSG.git
    os.chdir('CTNSG')

%pip install -r requirements.txt

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim

# Ensure local modules are reachable
sys.path.append(os.path.abspath('.'))

from macroplanner.gvt.model import GraphVQTransformer
from macroplanner.reldit.model import RelDiT
from orchestrator.arbor.planner import ArborPlanner
from realizer.realizer import CTNSGRealizer
from contracts.graph_schema import DiscourseGraph, SemanticNode, SemanticEdge

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

## 1. Load Preprocessed Supervisor Datasets (Module 2 & 4)
Loading the true DAGs, SDRT discourse trees, FAAP instruction pairs, and SAIGuard interaction topologies.

In [ ]:
import json
import os
from huggingface_hub import hf_hub_download

def fetch_and_load(filename):
    local_path = f'processed_data/{filename}'
    if not os.path.exists(local_path):
        print(f'Downloading {filename} from Hugging Face...')
        try:
            os.makedirs('processed_data', exist_ok=True)
            downloaded = hf_hub_download(repo_id='Borisz42/CTNSG-Graph-Curriculum', repo_type='dataset', filename=filename)
            import shutil
            shutil.copy(downloaded, local_path)
        except Exception as e:
            print(f'Warning: Could not download {filename}. {e}')
    
    if filename.endswith('.pt'):
        return torch.load(local_path) if os.path.exists(local_path) else []
    elif filename.endswith('.jsonl'):
        data = []
        if os.path.exists(local_path):
            with open(local_path, 'r', encoding='utf-8') as f:
                data = [json.loads(line) for line in f]
        return data

# Load FAAP (Seq2Seq)
faap_data = fetch_and_load('faap_instructions_full.jsonl')
print(f"Loaded {len(faap_data)} FAAP instruction pairs.")

# Load Topological Graphs
curriculum_graphs = fetch_and_load('ctnsg_curriculum.pt')
sdrt_graphs = fetch_and_load('sdrt_graphs_full.pt')
arbor_graphs = fetch_and_load('arbor_graphs_full.pt')
verification_graphs = fetch_and_load('verification_graphs_full.pt')

print(f"Loaded {len(curriculum_graphs)} Curriculum Graphs.")
print(f"Loaded {len(sdrt_graphs)} SDRT Discourse Graphs.")
print(f"Loaded {len(arbor_graphs)} Arbor TDP True DAGs.")
print(f"Loaded {len(verification_graphs)} SAIGuard/Brick Interaction Graphs.")

## 2. Initialize Global Orchestrator
Decoupling semantic and structural intent via Arbor.

In [ ]:
planner = ArborPlanner(input_dim=512, hidden_dim=256).to(device)
global_intent = torch.randn(1, 512).to(device)
decoupled_plan = planner.decouple_plan(global_intent)
print("Decoupled Plan Confidence:", decoupled_plan['confidence'])

## 3. GVT (Graph Vector Tokenization) Training Loop
Compressing continuous topologies into discrete graph tokens.

In [ ]:
import time
from datetime import datetime, timedelta

class GVTWrapper(nn.Module):
    def __init__(self, gvt):
        super().__init__()
        self.gvt = gvt
    def forward(self, nodes):
        empty_edges = torch.empty((2, 0), dtype=torch.long, device=nodes.device)
        return self.gvt(nodes, empty_edges)

num_gpus = torch.cuda.device_count()
if num_gpus > 0:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 22:
        base_bs = 8
    elif vram_gb >= 14:
        base_bs = 4
    elif vram_gb >= 7:
        base_bs = 2
    else:
        base_bs = 1
    batch_size = base_bs * num_gpus
else:
    vram_gb = 0
    batch_size = 2

print(f"Detected VRAM per GPU: {vram_gb:.1f} GB. Dynamically set global batch size to: {batch_size}")
max_seq = 1024

gvt_base = GraphVQTransformer(in_channels=256, hidden_channels=256, num_embeddings=64, num_quantizers=4).to(device)
if num_gpus > 1:
    gvt = nn.DataParallel(GVTWrapper(gvt_base))
else:
    gvt = GVTWrapper(gvt_base)

gvt_optimizer = optim.AdamW(gvt.parameters(), lr=3e-4)

print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting GVT Training Loop (Batch Size: {batch_size})...")
for epoch in range(3):
    epoch_start = time.time()
    total_loss_epoch = 0.0
    batched_nodes = []
    
    for graph in curriculum_graphs:
        nodes_full = graph['nodes'].to(device)
        for i in range(0, nodes_full.shape[0], max_seq):
            nodes = nodes_full[i : i + max_seq]
            if nodes.size(0) < max_seq:
                pad_size = max_seq - nodes.size(0)
                nodes = torch.cat([nodes, torch.zeros(pad_size, nodes.size(1), device=device)], dim=0)
                
            batched_nodes.append(nodes)
            
            if len(batched_nodes) == batch_size:
                nodes_tensor = torch.stack(batched_nodes, dim=0)
                gvt_optimizer.zero_grad()
                out = gvt(nodes_tensor)
                quantized_latents = out['z_q']
                vq_loss = out['commit_loss'].mean()
                recon_loss = nn.MSELoss()(quantized_latents, nodes_tensor)
                total_loss = recon_loss + vq_loss
                total_loss.backward()
                gvt_optimizer.step()
                total_loss_epoch += total_loss.item()
                batched_nodes = []
                
    if len(batched_nodes) > 0:
        nodes_tensor = torch.stack(batched_nodes, dim=0)
        gvt_optimizer.zero_grad()
        out = gvt(nodes_tensor)
        quantized_latents = out['z_q']
        vq_loss = out['commit_loss'].mean()
        recon_loss = nn.MSELoss()(quantized_latents, nodes_tensor)
        total_loss = recon_loss + vq_loss
        total_loss.backward()
        gvt_optimizer.step()
        total_loss_epoch += total_loss.item()
        
    epoch_duration = time.time() - epoch_start
    avg_loss = total_loss_epoch / max(1, len(curriculum_graphs) // batch_size)
    eta_seconds = epoch_duration * (3 - (epoch + 1))
    eta_str = str(timedelta(seconds=int(eta_seconds)))
    timestamp = datetime.now().strftime('%H:%M:%S')
    print(f"[{timestamp}] Epoch {epoch+1}/3 | GVT Loss: {avg_loss:.4f} | Time: {epoch_duration:.1f}s | ETA: {eta_str}")

print("Saving GVT weights before starting RelDiT...")
import os
os.makedirs("ctnsg_export", exist_ok=True)
if num_gpus > 1:
    torch.save(gvt.module.gvt.state_dict(), "ctnsg_export/gvt_weights.pt")
else:
    torch.save(gvt.gvt.state_dict(), "ctnsg_export/gvt_weights.pt")


## 4. RelDiT (Relational Diffusion) Training Loop
Training the discrete diffusion model to generate topologies.

In [ ]:
# Data Splitting for Validation
val_split_size = min(200, len(curriculum_graphs) // 10)
train_graphs = curriculum_graphs[:-val_split_size]
val_graphs = curriculum_graphs[-val_split_size:]

def cache_graphs(graphs, gvt_model, bs):
    gvt_model.eval()
    all_tokens = []
    batched_nodes = []
    import torch
    with torch.no_grad():
        for graph in graphs:
            nodes_full = graph['nodes'].to(device)
            for i in range(0, nodes_full.shape[0], max_seq):
                nodes = nodes_full[i : i + max_seq]
                if nodes.size(0) < max_seq:
                    nodes = torch.cat([nodes, torch.zeros(max_seq - nodes.size(0), nodes.size(1), device=device)], dim=0)
                batched_nodes.append(nodes)
                if len(batched_nodes) == bs:
                    nodes_tensor = torch.stack(batched_nodes, dim=0)
                    out = gvt_model(nodes_tensor)
                    tokens = out['discrete_tokens'][:, :, 0]
                    all_tokens.append(tokens.cpu())
                    batched_nodes = []
        if len(batched_nodes) > 0:
            nodes_tensor = torch.stack(batched_nodes, dim=0)
            out = gvt_model(nodes_tensor)
            tokens = out['discrete_tokens'][:, :, 0]
            all_tokens.append(tokens.cpu())
    if all_tokens:
        return torch.cat(all_tokens, dim=0)
    return torch.empty((0, max_seq), dtype=torch.long)

print("Pre-computing all discrete tokens...")
eval_batch_size = batch_size * 2
cached_train_dataset = cache_graphs(train_graphs, gvt, eval_batch_size)
cached_val_dataset = cache_graphs(val_graphs, gvt, eval_batch_size)
print(f"Caching complete! Train shape: {cached_train_dataset.shape}, Val shape: {cached_val_dataset.shape}")

print(f"Data split: {len(train_graphs)} training graphs, {len(val_graphs)} validation graphs.")

reldit_base = RelDiT(vocab_size=64, d_model=256).to(device)
if num_gpus > 1:
    reldit = nn.DataParallel(reldit_base)
else:
    reldit = reldit_base

reldit_optimizer = optim.AdamW(reldit.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler() # Mixed Precision Scaling

best_val_loss = float('inf')
patience = 10
patience_counter = 0

print("Caching training tokens for V.U.N Novelty tracking...")
train_token_cache = set()
for i in range(len(cached_train_dataset)):
    tokens_np = cached_train_dataset[i].numpy().tobytes()
    train_token_cache.add(tokens_np)

# Dynamic batch size for RelDiT (smaller model, much larger batch size)
if num_gpus > 0:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 22:
        reldit_base_bs = 64
    elif vram_gb >= 14:
        reldit_base_bs = 32
    elif vram_gb >= 7:
        reldit_base_bs = 16
    else:
        reldit_base_bs = 8
    reldit_batch_size = reldit_base_bs * num_gpus
else:
    reldit_batch_size = 8

# MASSIVE OPTIMIZATION: Move dataset entirely to GPU if it fits (~160MB for 20k graphs)
try:
    cached_train_dataset = cached_train_dataset.to(device)
    cached_val_dataset = cached_val_dataset.to(device)
    print("Moved cached datasets entirely to GPU VRAM for maximum speed.")
except RuntimeError:
    print("Not enough VRAM to keep entire dataset on GPU, leaving on CPU.")

print()
print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting RelDiT Training Loop (Batch Size: {reldit_batch_size})...")
num_epochs = 100
reldit_start_time = time.time()
max_reldit_time = 11 * 3600

for epoch in range(num_epochs):
    epoch_start = time.time()
    if time.time() - reldit_start_time > max_reldit_time:
        print()
        print(f"[{datetime.now().strftime('%H:%M:%S')}] 11-hour time limit reached! Halting RelDiT training safely.")
        reldit.load_state_dict(torch.load('best_reldit_weights.pt'))
        break
        
    reldit.train()
    total_loss_epoch = 0.0
    import torch
    # Randperm on device to avoid CPU-GPU syncs
    indices = torch.randperm(len(cached_train_dataset), device=device if cached_train_dataset.is_cuda else 'cpu')
    
    for i in range(0, len(cached_train_dataset), reldit_batch_size):
        batch_indices = indices[i:i + reldit_batch_size]
        # If already on GPU, this is practically instant
        token_batch = cached_train_dataset[batch_indices].to(device)
        
        reldit_optimizer.zero_grad(set_to_none=True)
        
        # AMP Mixed Precision Forward Pass
        with torch.cuda.amp.autocast():
            loss = reldit(token_batch)
            loss = loss.mean()
            
        scaler.scale(loss).backward()
        scaler.step(reldit_optimizer)
        scaler.update()
        
        total_loss_epoch += loss.item()
        
    avg_train_loss = total_loss_epoch / max(1, len(cached_train_dataset) // reldit_batch_size)
    
    reldit.eval()
    val_loss_epoch = 0.0
    with torch.no_grad():
        for i in range(0, len(cached_val_dataset), reldit_batch_size):
            token_batch = cached_val_dataset[i:i + reldit_batch_size].to(device)
            with torch.cuda.amp.autocast():
                loss = reldit(token_batch)
                loss = loss.mean()
            val_loss_epoch += loss.item()
            
    avg_val_loss = val_loss_epoch / max(1, len(cached_val_dataset) // reldit_batch_size)
    
    epoch_duration = time.time() - epoch_start
    eta_seconds = epoch_duration * (num_epochs - (epoch + 1))
    from datetime import timedelta
    eta_str = str(timedelta(seconds=int(eta_seconds)))
    timestamp = datetime.now().strftime('%H:%M:%S')
    print(f"[{timestamp}] Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {epoch_duration:.1f}s | ETA: {eta_str}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(reldit_base.state_dict(), 'best_reldit_weights.pt')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print()
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Early stopping! Best Val Loss: {best_val_loss:.4f}")
            reldit_base.load_state_dict(torch.load('best_reldit_weights.pt'))
            break
            
    if (epoch + 1) % 5 == 0:
        print(f"  -> Evaluating V.U.N Metrics (Epoch {epoch+1})...")
        N_samples = 10
        gen_seq_len = 1024
        with torch.no_grad():
            gen_tokens = reldit_base.generate(batch_size=N_samples, seq_len=gen_seq_len, device=device, use_critic=False)
            scaled_logits = reldit_base.transformer(gen_tokens) / 1.0
            probs = torch.softmax(scaled_logits, dim=-1)
            pred_probs = torch.gather(probs, -1, gen_tokens.unsqueeze(-1)).squeeze(-1)
            validity = pred_probs.mean().item() * 100.0
            unique_seqs = set([tuple(seq.tolist()) for seq in gen_tokens])
            uniqueness = (len(unique_seqs) / N_samples) * 100.0
            novel_count = sum(1 for seq in unique_seqs if torch.tensor(seq, dtype=torch.long).numpy().tobytes() not in train_token_cache)
            novelty = (novel_count / max(1, len(unique_seqs))) * 100.0
        print(f"  -> [V.U.N] Validity: {validity:.1f}% | Uniqueness: {uniqueness:.1f}% | Novelty: {novelty:.1f}%")

## 5. CID Critic Training Loop
Training the Post-Hoc CID Critic.

In [ ]:
print()
print('======================================================')
print('Phase 3: Critical Iterative Denoising (CID) Post-Hoc Critic Training')
print('======================================================')
print()
print('Freezing the main RelDiT Denoiser parameters...')

for name, param in reldit_base.named_parameters():
    if 'critic' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

critic_optimizer = optim.AdamW(reldit_base.critic.parameters(), lr=1e-4)
bce_loss = nn.BCELoss()
critic_scaler = torch.cuda.amp.GradScaler()

print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting Critic Training Loop...")
num_critic_epochs = 10
critic_train_tokens = cached_train_dataset[:500]

for epoch in range(num_critic_epochs):
    epoch_start = time.time()
    reldit.train()
    total_loss_epoch = 0.0
    import torch
    indices = torch.randperm(len(critic_train_tokens), device=device if critic_train_tokens.is_cuda else 'cpu')
    for i in range(0, len(critic_train_tokens), reldit_batch_size):
        batch_indices = indices[i:i + reldit_batch_size]
        x0 = critic_train_tokens[batch_indices].to(device)
                    
        t = torch.randint(1, reldit_base.num_timesteps + 1, (x0.size(0),), device=device)
        x_t, mask = reldit_base.diffusion.add_noise(x0, t)
        
        critic_optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            logits = reldit_base.transformer(x_t)
            probs = torch.softmax(logits, dim=-1)
            critic_scores = reldit_base.get_critic_scores(probs, t)
            target = (~mask).float()
            loss = bce_loss(critic_scores, target)
            
        critic_scaler.scale(loss).backward()
        critic_scaler.step(critic_optimizer)
        critic_scaler.update()
        total_loss_epoch += loss.item()
                
    avg_train_loss = total_loss_epoch / max(1, len(critic_train_tokens) // reldit_batch_size)
    epoch_duration = time.time() - epoch_start
    timestamp = datetime.now().strftime('%H:%M:%S')
    print(f"[{timestamp}] Epoch {epoch+1}/{num_critic_epochs} | Critic BCE Loss: {avg_train_loss:.4f} | Time: {epoch_duration:.1f}s")

print()
print('Critic training complete! The CID framework is now fully integrated.')
print('To generate highly valid graph topologies, ensure use_critic=True is passed to generate()')

## 5. Realizer Inference Pipeline
Integrating the VNPool, O(1) GREATGRAMMA enforcement, and SafeLLM to generate text.

In [ ]:
print()
print("Initializing Realizer Pipeline...")
realizer = CTNSGRealizer()

# Create a stub discourse graph to feed into the Realizer
inference_graph = DiscourseGraph(
    graph_id="infer_001",
    nodes=[
        SemanticNode(node_id="n1", concept="System", vq_index=12),
        SemanticNode(node_id="n2", concept="Macroplanner", vq_index=45)
    ],
    edges=[
        SemanticEdge(source_id="n1", target_id="n2", relation_type="triggers")
    ]
)

schema = {"type": "object", "properties": {"output": {"type": "string"}}}
result = realizer.generate(inference_graph, schema, context_lines=5)

print()
print("=== Realizer Final Output ===")
print(result)

## 6. Model Weight Export & Distribution
Saves the trained components (GVT and RelDiT) to /kaggle/working/ and packages them into a .zip file. This zip file can be easily downloaded or natively linked as a 'Notebook Output' dataset into the Evaluation Suite notebook without needing manual downloads.

In [ ]:
import shutil

print()
print('Saving trained models...')
os.makedirs('ctnsg_export', exist_ok=True)
torch.save(gvt.state_dict(), 'ctnsg_export/gvt_weights.pt')
torch.save(reldit.state_dict(), 'ctnsg_export/reldit_weights.pt')

# Create an export manifest
manifest = {
    'architecture': 'CTNSG',
    'modules': ['GVT', 'RelDiT'],
    'dim': 256
}
with open('ctnsg_export/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=4)

# Zip the export directory
print('Packaging into ctnsg_release.zip...')
shutil.make_archive('/kaggle/working/ctnsg_release', 'zip', 'ctnsg_export')

print('Export complete! ctnsg_release.zip is now available in the Kaggle /kaggle/working/ directory.')